# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [4]:
# TODO: Load environment variables
load_dotenv()

True

### VectorDB Instance

In [5]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL")
)

In [7]:
# TODO: Create a collection
# Choose any name you want
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [10]:
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

    print(f"Added: {file_name} -> {game['Name']} ({game['Platform']})")

print(f"\nTotal games indexed: {collection.count()}")

Added: 001.json -> Gran Turismo (PlayStation 1)
Added: 002.json -> Grand Theft Auto: San Andreas (PlayStation 2)
Added: 003.json -> Gran Turismo 5 (PlayStation 3)
Added: 004.json -> Marvel's Spider-Man (PlayStation 4)
Added: 005.json -> Marvel's Spider-Man 2 (PlayStation 5)
Added: 006.json -> Pokémon Gold and Silver (Game Boy Color)
Added: 007.json -> Pokémon Ruby and Sapphire (Game Boy Advance)
Added: 008.json -> Super Mario World (Super Nintendo Entertainment System (SNES))
Added: 009.json -> Super Mario 64 (Nintendo 64)
Added: 010.json -> Super Smash Bros. Melee (GameCube)
Added: 011.json -> Wii Sports (Wii)
Added: 012.json -> Mario Kart 8 Deluxe (Nintendo Switch)
Added: 013.json -> Kinect Adventures! (Xbox 360)
Added: 014.json -> Minecraft (Xbox One)
Added: 015.json -> Halo Infinite (Xbox Series X|S)

Total games indexed: 15


In [11]:
results = collection.query(
    query_texts=["Mario platformer game Nintendo"],
    n_results=5,
    include=["documents", "metadatas", "distances"]
  )

for i in range(len(results["documents"][0])):
  print(f"Rank {i+1}")
  print(f"  Name: {results['metadatas'][0][i].get('Name')}")
  print(f"  Platform: {results['metadatas'][0][i].get('Platform')}")
  print(f"  Year: {results['metadatas'][0][i].get('YearOfRelease')}")
  print(f"  Distance: {results['distances'][0][i]:.4f}")
  print()

Rank 1
  Name: Super Mario 64
  Platform: Nintendo 64
  Year: 1996
  Distance: 0.2158

Rank 2
  Name: Super Mario World
  Platform: Super Nintendo Entertainment System (SNES)
  Year: 1990
  Distance: 0.2453

Rank 3
  Name: Mario Kart 8 Deluxe
  Platform: Nintendo Switch
  Year: 2017
  Distance: 0.3172

Rank 4
  Name: Super Smash Bros. Melee
  Platform: GameCube
  Year: 2001
  Distance: 0.3462

Rank 5
  Name: Pokémon Gold and Silver
  Platform: Game Boy Color
  Year: 1999
  Distance: 0.3854



In [13]:
import shutil
shutil.make_archive('chromadb_backup', 'zip', '.', 'chromadb')

'/workspace/Code/project/starter/chromadb_backup.zip'